# Segmentation parameter sweep

In [ ]:
from __future__ import annotations

import json
import logging
import math
import sys
import time
from copy import deepcopy
from datetime import datetime, timezone
from pathlib import Path
from typing import Any, Dict, List, Tuple

import numpy as np
import pandas as pd
import yaml
from scipy.ndimage import uniform_filter1d
from scipy.signal import find_peaks
from scipy.spatial.distance import cosine
from tqdm.auto import tqdm

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 240)

In [ ]:
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

print("PROJECT_ROOT =", PROJECT_ROOT)

CONFIG_PATH = PROJECT_ROOT / "configs" / "config.yaml"
print("CONFIG_PATH  =", CONFIG_PATH)

DEFAULT_CONFIG = {
    "paths": {
        "output_dir": "output",
        "log_dir": "output/logs",
    },
    "ssl": {
        "output_subdir": "01_ssl_frames",
    },
    "segmentation": {
        "input_frame_index_csv": "output/01_ssl_frames/frame_index.csv",
        "input_ssl_subdir": "output/01_ssl_frames",
        "output_subdir": "02_segments",
        "overwrite": False,
        "max_clips": None,
        "smoothing_window_frames": 5,
        "change_score_smoothing_frames": 3,
        "distance_metric": "cosine",
        "peak_prominence": 0.025,
        "peak_height_mad_multiplier": 3.0,
        "min_peak_distance_sec": 0.20,
        "min_segment_duration_sec": 0.25,
        "max_segments_per_clip": None,
        "save_debug_scores": False,
    },
    "segment_sweep": {
        "output_subdir": "02b_segment_sweep",
        "overwrite_candidates": False,
        "candidate_prefix": "candidate_",
        "run_downstream_in_02b": False,
        "default_candidates": [
            {
                "candidate_id": "candidate_A",
                "candidate_label": "gentle_split",
                "notes": "Slightly more sensitive than baseline while keeping strong merge protection.",
                "smoothing_window_frames": 5,
                "change_score_smoothing_frames": 3,
                "peak_prominence": 0.022,
                "peak_height_mad_multiplier": 2.75,
                "min_peak_distance_sec": 0.18,
                "min_segment_duration_sec": 0.20,
                "max_segments_per_clip": None,
            },
            {
                "candidate_id": "candidate_B",
                "candidate_label": "balanced_split",
                "notes": "Primary balanced candidate for reducing single-segment dominance without fragment explosion.",
                "smoothing_window_frames": 5,
                "change_score_smoothing_frames": 2,
                "peak_prominence": 0.020,
                "peak_height_mad_multiplier": 2.50,
                "min_peak_distance_sec": 0.15,
                "min_segment_duration_sec": 0.18,
                "max_segments_per_clip": None,
            },
            {
                "candidate_id": "candidate_C",
                "candidate_label": "richer_sequences",
                "notes": "Most sensitive candidate in the first sweep, still bounded by minimum duration.",
                "smoothing_window_frames": 4,
                "change_score_smoothing_frames": 2,
                "peak_prominence": 0.018,
                "peak_height_mad_multiplier": 2.25,
                "min_peak_distance_sec": 0.14,
                "min_segment_duration_sec": 0.16,
                "max_segments_per_clip": 12,
            },
        ],
    },
}

def deep_update(base: Dict[str, Any], updates: Dict[str, Any]) -> Dict[str, Any]:
    result = deepcopy(base)
    for key, value in (updates or {}).items():
        if isinstance(value, dict) and isinstance(result.get(key), dict):
            result[key] = deep_update(result[key], value)
        else:
            result[key] = value
    return result

CONFIG = deepcopy(DEFAULT_CONFIG)
if CONFIG_PATH.exists():
    with open(CONFIG_PATH, "r", encoding="utf-8") as f:
        loaded = yaml.safe_load(f) or {}
    CONFIG = deep_update(CONFIG, loaded)

def resolve_path(raw_path: str) -> Path:
    path = Path(raw_path)
    if path.is_absolute():
        return path
    return (PROJECT_ROOT / path).resolve()

OUTPUT_ROOT = resolve_path(CONFIG["paths"]["output_dir"])
LOG_DIR = resolve_path(CONFIG["paths"]["log_dir"])
LOG_DIR.mkdir(parents=True, exist_ok=True)

SSL_CFG = CONFIG["ssl"]
SEG_CFG = CONFIG["segmentation"]
SWEEP_CFG = CONFIG["segment_sweep"]

FRAME_INDEX_PATH = resolve_path(SEG_CFG["input_frame_index_csv"])
SSL_DIR = resolve_path(SEG_CFG["input_ssl_subdir"])
BASELINE_SEGMENTS_DIR = OUTPUT_ROOT / str(SEG_CFG["output_subdir"]).strip("/")
SWEEP_DIR = OUTPUT_ROOT / str(SWEEP_CFG["output_subdir"]).strip("/")

SWEEP_DIR.mkdir(parents=True, exist_ok=True)

BASELINE_SEGMENTS_CSV = BASELINE_SEGMENTS_DIR / "segments.csv"
BASELINE_SUMMARY_CSV = BASELINE_SEGMENTS_DIR / "segment_summary.csv"

SWEEP_PLAN_CSV = SWEEP_DIR / "sweep_plan.csv"
SWEEP_RESULTS_CSV = SWEEP_DIR / "sweep_results.csv"
BASELINE_REFERENCE_JSON = SWEEP_DIR / "baseline_reference_metrics.json"

print("FRAME_INDEX_PATH      =", FRAME_INDEX_PATH)
print("SSL_DIR               =", SSL_DIR)
print("BASELINE_SEGMENTS_DIR =", BASELINE_SEGMENTS_DIR)
print("SWEEP_DIR             =", SWEEP_DIR)

In [ ]:
LOG_FILE = LOG_DIR / "02b_segment_sweep.log"

logger = logging.getLogger("stage4_segment_sweep")
logger.setLevel(logging.INFO)
logger.handlers.clear()

formatter = logging.Formatter("%(asctime)s | %(levelname)s | %(message)s")

file_handler = logging.FileHandler(LOG_FILE, encoding="utf-8")
file_handler.setFormatter(formatter)
logger.addHandler(file_handler)

stream_handler = logging.StreamHandler(sys.stdout)
stream_handler.setFormatter(formatter)
logger.addHandler(stream_handler)

logger.info("Logging initialized")
logger.info("Log file: %s", LOG_FILE)

In [ ]:
def sanitize_clip_id(value: str) -> str:
    text = str(value).strip()
    return text.replace("/", "_").replace("\\", "_").replace(" ", "_")

def load_frame_index(path: Path) -> pd.DataFrame:
    if not path.exists():
        raise FileNotFoundError(f"Frame index file not found: {path}")
    df = pd.read_csv(path)
    required = {"clip_id", "frame_file", "duration_sec", "num_frames", "frame_hop_sec", "status"}
    missing = required - set(df.columns)
    if missing:
        raise ValueError(f"frame_index.csv is missing required columns: {sorted(missing)}")
    df = df.copy()
    df["clip_id"] = df["clip_id"].astype(str).map(sanitize_clip_id)
    df["frame_file"] = df["frame_file"].astype(str)
    return df

def safe_float(value: Any, default: float = 0.0) -> float:
    try:
        if pd.isna(value):
            return default
        return float(value)
    except Exception:
        return default

def robust_mad(x: np.ndarray) -> float:
    x = np.asarray(x, dtype=np.float32)
    if x.size == 0:
        return 0.0
    med = np.median(x)
    mad = np.median(np.abs(x - med))
    return float(mad)

def load_ssl_npz(npz_path: Path, fallback_duration_sec: float, fallback_frame_hop_sec: float) -> Dict[str, Any]:
    if not npz_path.exists():
        raise FileNotFoundError(f"Missing SSL frame file: {npz_path}")

    with np.load(npz_path, allow_pickle=False) as data:
        embeddings = data["embeddings"].astype(np.float32, copy=False)

        if "frame_times_sec" in data.files:
            frame_times_sec = data["frame_times_sec"].astype(np.float32, copy=False)
        else:
            hop = float(np.asarray(data["frame_hop_sec"]).reshape(-1)[0]) if "frame_hop_sec" in data.files else float(fallback_frame_hop_sec)
            frame_times_sec = (np.arange(embeddings.shape[0], dtype=np.float32) * np.float32(hop)).astype(np.float32)

        duration_sec = float(np.asarray(data["duration_sec"]).reshape(-1)[0]) if "duration_sec" in data.files else float(fallback_duration_sec)
        frame_hop_sec = float(np.asarray(data["frame_hop_sec"]).reshape(-1)[0]) if "frame_hop_sec" in data.files else float(fallback_frame_hop_sec)

    if embeddings.ndim != 2:
        raise ValueError(f"Expected embeddings to be 2D, got shape={embeddings.shape} for {npz_path}")

    if frame_times_sec.shape[0] != embeddings.shape[0]:
        if embeddings.shape[0] > 0:
            frame_times_sec = np.linspace(0.0, max(duration_sec - frame_hop_sec, 0.0), embeddings.shape[0], dtype=np.float32)
        else:
            frame_times_sec = np.array([], dtype=np.float32)

    return {
        "embeddings": embeddings,
        "frame_times_sec": frame_times_sec,
        "duration_sec": float(duration_sec),
        "frame_hop_sec": float(frame_hop_sec),
    }

def smooth_embeddings(embeddings: np.ndarray, window_frames: int) -> np.ndarray:
    window_frames = max(int(window_frames), 1)
    if embeddings.shape[0] <= 1 or window_frames <= 1:
        return embeddings.astype(np.float32, copy=False)
    return uniform_filter1d(embeddings.astype(np.float32, copy=False), size=window_frames, axis=0, mode="nearest")

def compute_change_scores(embeddings_smoothed: np.ndarray, metric: str = "cosine") -> np.ndarray:
    T = embeddings_smoothed.shape[0]
    if T <= 1:
        return np.zeros((T,), dtype=np.float32)

    prev = embeddings_smoothed[:-1]
    curr = embeddings_smoothed[1:]

    if metric == "cosine":
        prev_norm = np.linalg.norm(prev, axis=1, keepdims=True)
        curr_norm = np.linalg.norm(curr, axis=1, keepdims=True)
        prev_norm = np.maximum(prev_norm, 1e-8)
        curr_norm = np.maximum(curr_norm, 1e-8)
        similarity = np.sum((prev / prev_norm) * (curr / curr_norm), axis=1)
        similarity = np.clip(similarity, -1.0, 1.0)
        change = 1.0 - similarity
    elif metric == "euclidean":
        change = np.linalg.norm(curr - prev, axis=1)
    else:
        raise ValueError(f"Unsupported distance metric: {metric}")

    scores = np.zeros((T,), dtype=np.float32)
    scores[1:] = change.astype(np.float32, copy=False)
    return scores

def smooth_scores(scores: np.ndarray, window_frames: int) -> np.ndarray:
    window_frames = max(int(window_frames), 1)
    if scores.size <= 1 or window_frames <= 1:
        return scores.astype(np.float32, copy=False)
    return uniform_filter1d(scores.astype(np.float32, copy=False), size=window_frames, mode="nearest")

def detect_boundary_frames(
    scores: np.ndarray,
    frame_hop_sec: float,
    peak_prominence: float,
    peak_height_mad_multiplier: float,
    min_peak_distance_sec: float,
    max_segments_per_clip: int | None = None,
) -> Tuple[np.ndarray, Dict[str, float]]:
    if scores.size <= 2:
        return np.array([], dtype=np.int32), {
            "median_score": float(np.median(scores)) if scores.size else 0.0,
            "mad_score": robust_mad(scores),
            "peak_height_threshold": np.inf,
        }

    median_score = float(np.median(scores))
    mad_score = robust_mad(scores)
    adaptive_height = median_score + float(peak_height_mad_multiplier) * mad_score
    min_distance_frames = max(int(round(float(min_peak_distance_sec) / max(float(frame_hop_sec), 1e-6))), 1)

    peaks, properties = find_peaks(
        scores,
        height=adaptive_height,
        prominence=float(peak_prominence),
        distance=min_distance_frames,
    )

    peaks = peaks.astype(np.int32, copy=False)

    if max_segments_per_clip is not None and len(peaks) > 0:
        max_boundaries = max(int(max_segments_per_clip) - 1, 0)
        if len(peaks) > max_boundaries:
            prominences = properties.get("prominences", np.zeros_like(peaks, dtype=np.float32))
            order = np.argsort(prominences)[::-1][:max_boundaries]
            peaks = np.sort(peaks[order])

    return peaks, {
        "median_score": median_score,
        "mad_score": float(mad_score),
        "peak_height_threshold": float(adaptive_height),
    }

def boundaries_to_segments(num_frames: int, boundary_frames: np.ndarray) -> List[Tuple[int, int]]:
    boundaries = sorted(int(b) for b in boundary_frames.tolist() if 0 < int(b) < num_frames)
    starts = [0] + boundaries
    ends = boundaries + [num_frames]
    segments = []
    for start, end in zip(starts, ends):
        if end > start:
            segments.append((int(start), int(end)))
    return segments

def cosine_distance_safe(a: np.ndarray, b: np.ndarray) -> float:
    a = np.asarray(a, dtype=np.float32)
    b = np.asarray(b, dtype=np.float32)
    if a.size == 0 or b.size == 0:
        return np.inf
    a_norm = np.linalg.norm(a)
    b_norm = np.linalg.norm(b)
    if a_norm < 1e-8 or b_norm < 1e-8:
        return np.inf
    return float(cosine(a, b))

def merge_short_segments(
    embeddings_smoothed: np.ndarray,
    segments: List[Tuple[int, int]],
    frame_hop_sec: float,
    min_segment_duration_sec: float,
) -> List[Tuple[int, int]]:
    if not segments:
        return []

    min_frames = max(int(math.ceil(float(min_segment_duration_sec) / max(float(frame_hop_sec), 1e-6))), 1)
    segments = [(int(s), int(e)) for s, e in segments if e > s]

    changed = True
    while changed and len(segments) > 1:
        changed = False
        for idx, (start, end) in enumerate(list(segments)):
            length = end - start
            if length >= min_frames:
                continue

            if idx == 0:
                next_start, next_end = segments[idx + 1]
                segments[idx + 1] = (start, next_end)
                del segments[idx]
                changed = True
                break
            elif idx == len(segments) - 1:
                prev_start, prev_end = segments[idx - 1]
                segments[idx - 1] = (prev_start, end)
                del segments[idx]
                changed = True
                break
            else:
                prev_start, prev_end = segments[idx - 1]
                next_start, next_end = segments[idx + 1]

                short_mean = embeddings_smoothed[start:end].mean(axis=0)
                prev_mean = embeddings_smoothed[prev_start:prev_end].mean(axis=0)
                next_mean = embeddings_smoothed[next_start:next_end].mean(axis=0)

                prev_dist = cosine_distance_safe(short_mean, prev_mean)
                next_dist = cosine_distance_safe(short_mean, next_mean)

                if prev_dist <= next_dist:
                    segments[idx - 1] = (prev_start, end)
                    del segments[idx]
                else:
                    segments[idx + 1] = (start, next_end)
                    del segments[idx]
                changed = True
                break

    segments = [(int(s), int(e)) for s, e in segments if e > s]
    return segments

def summarize_scores(scores: np.ndarray) -> Dict[str, float]:
    if scores.size == 0:
        return {
            "score_mean": 0.0,
            "score_std": 0.0,
            "score_max": 0.0,
        }
    return {
        "score_mean": float(np.mean(scores)),
        "score_std": float(np.std(scores)),
        "score_max": float(np.max(scores)),
    }

def build_segment_rows(
    clip_id: str,
    frame_file: str,
    duration_sec: float,
    frame_hop_sec: float,
    frame_times_sec: np.ndarray,
    segments: List[Tuple[int, int]],
) -> List[Dict[str, Any]]:
    rows: List[Dict[str, Any]] = []
    total_frames = len(frame_times_sec)

    for idx, (start_frame, end_frame) in enumerate(segments):
        start_frame = int(start_frame)
        end_frame = int(end_frame)
        if end_frame <= start_frame:
            continue

        start_sec = float(frame_times_sec[start_frame]) if start_frame < total_frames else float(start_frame * frame_hop_sec)
        if end_frame - 1 < total_frames:
            end_sec = float(frame_times_sec[end_frame - 1] + frame_hop_sec)
        else:
            end_sec = float(end_frame * frame_hop_sec)

        end_sec = min(end_sec, float(duration_sec))
        duration = max(end_sec - start_sec, 0.0)

        rows.append({
            "clip_id": clip_id,
            "segment_id": f"{clip_id}__seg_{idx:04d}",
            "segment_index": idx,
            "frame_file": frame_file,
            "start_frame": start_frame,
            "end_frame_exclusive": end_frame,
            "num_frames": int(end_frame - start_frame),
            "start_sec": float(start_sec),
            "end_sec": float(end_sec),
            "duration_sec": float(duration),
            "clip_duration_sec": float(duration_sec),
            "frame_hop_sec": float(frame_hop_sec),
        })
    return rows

def build_summary_row(
    clip_id: str,
    frame_file: str,
    duration_sec: float,
    num_total_frames: int,
    frame_hop_sec: float,
    boundary_frames_initial: np.ndarray,
    segments_final: List[Tuple[int, int]],
    score_meta: Dict[str, float],
    score_summary: Dict[str, float],
) -> Dict[str, Any]:
    segment_lengths_frames = np.array([e - s for s, e in segments_final], dtype=np.int32) if segments_final else np.array([], dtype=np.int32)
    segment_lengths_sec = segment_lengths_frames.astype(np.float32) * np.float32(frame_hop_sec) if segment_lengths_frames.size else np.array([], dtype=np.float32)

    return {
        "clip_id": clip_id,
        "frame_file": frame_file,
        "clip_duration_sec": float(duration_sec),
        "num_total_frames": int(num_total_frames),
        "frame_hop_sec": float(frame_hop_sec),
        "initial_boundary_count": int(len(boundary_frames_initial)),
        "final_segment_count": int(len(segments_final)),
        "median_segment_duration_sec": float(np.median(segment_lengths_sec)) if segment_lengths_sec.size else float(duration_sec),
        "min_segment_duration_sec": float(np.min(segment_lengths_sec)) if segment_lengths_sec.size else float(duration_sec),
        "max_segment_duration_sec": float(np.max(segment_lengths_sec)) if segment_lengths_sec.size else float(duration_sec),
        "median_score": float(score_meta.get("median_score", 0.0)),
        "mad_score": float(score_meta.get("mad_score", 0.0)),
        "peak_height_threshold": float(score_meta.get("peak_height_threshold", 0.0)),
        "score_mean": float(score_summary.get("score_mean", 0.0)),
        "score_std": float(score_summary.get("score_std", 0.0)),
        "score_max": float(score_summary.get("score_max", 0.0)),
        "processed_at_utc": datetime.now(timezone.utc).isoformat(),
    }

def summarize_candidate_outputs(segments_df: pd.DataFrame, summary_df: pd.DataFrame, candidate_params: Dict[str, Any]) -> Dict[str, Any]:
    out = dict(candidate_params)
    out["n_clips"] = int(summary_df["clip_id"].nunique()) if not summary_df.empty else 0
    out["n_segments_total"] = int(len(segments_df))
    out["mean_segments_per_clip"] = float(summary_df["final_segment_count"].mean()) if not summary_df.empty else np.nan
    out["median_segments_per_clip"] = float(summary_df["final_segment_count"].median()) if not summary_df.empty else np.nan
    out["single_segment_clips"] = int((summary_df["final_segment_count"] == 1).sum()) if not summary_df.empty else 0
    out["single_segment_clip_pct"] = float((summary_df["final_segment_count"] == 1).mean() * 100.0) if not summary_df.empty else np.nan
    out["mean_initial_boundary_count"] = float(summary_df["initial_boundary_count"].mean()) if not summary_df.empty else np.nan
    out["median_initial_boundary_count"] = float(summary_df["initial_boundary_count"].median()) if not summary_df.empty else np.nan
    out["median_segment_duration_sec_over_clips"] = float(summary_df["median_segment_duration_sec"].median()) if not summary_df.empty else np.nan
    out["min_segment_duration_sec_overall"] = float(summary_df["min_segment_duration_sec"].min()) if not summary_df.empty else np.nan
    out["max_segment_duration_sec_overall"] = float(summary_df["max_segment_duration_sec"].max()) if not summary_df.empty else np.nan

    if not segments_df.empty:
        seg_dur = segments_df["duration_sec"].astype(float)
        out["p10_segment_duration_sec"] = float(np.percentile(seg_dur, 10))
        out["p50_segment_duration_sec"] = float(np.percentile(seg_dur, 50))
        out["p90_segment_duration_sec"] = float(np.percentile(seg_dur, 90))
        out["tiny_segments_lt_015_sec"] = int((seg_dur < 0.15).sum())
        out["tiny_segments_lt_020_sec"] = int((seg_dur < 0.20).sum())
        out["tiny_segments_lt_015_pct"] = float((seg_dur < 0.15).mean() * 100.0)
        out["tiny_segments_lt_020_pct"] = float((seg_dur < 0.20).mean() * 100.0)
        out["max_segments_in_any_clip"] = int(summary_df["final_segment_count"].max()) if not summary_df.empty else np.nan
    else:
        out["p10_segment_duration_sec"] = np.nan
        out["p50_segment_duration_sec"] = np.nan
        out["p90_segment_duration_sec"] = np.nan
        out["tiny_segments_lt_015_sec"] = 0
        out["tiny_segments_lt_020_sec"] = 0
        out["tiny_segments_lt_015_pct"] = np.nan
        out["tiny_segments_lt_020_pct"] = np.nan
        out["max_segments_in_any_clip"] = np.nan

    return out

def ensure_candidate_dirs(candidate_dir: Path) -> Dict[str, Path]:
    paths = {
        "root": candidate_dir,
        "segments_csv": candidate_dir / "segments.csv",
        "segment_summary_csv": candidate_dir / "segment_summary.csv",
        "candidate_config_json": candidate_dir / "candidate_config.json",
        "candidate_metrics_json": candidate_dir / "candidate_metrics.json",
        "notes_md": candidate_dir / "candidate_notes.md",
        "segment_vectors_dir": candidate_dir / "03_segment_vectors",
        "kstudy_dir": candidate_dir / "04_kstudy",
        "tokens_dir": candidate_dir / "05_tokens",
        "dictionary_dir": candidate_dir / "06_dictionary",
        "audits_dir": candidate_dir / "07_audits",
    }
    for key, p in paths.items():
        if key.endswith("_dir") or key == "root":
            p.mkdir(parents=True, exist_ok=True)
    return paths

def write_markdown(path: Path, lines: List[str]) -> None:
    path.write_text("\n".join(lines).strip() + "\n", encoding="utf-8")

def run_single_candidate(work_df: pd.DataFrame, candidate_params: Dict[str, Any]) -> Tuple[pd.DataFrame, pd.DataFrame]:
    segment_rows: List[Dict[str, Any]] = []
    summary_rows: List[Dict[str, Any]] = []

    for row in tqdm(work_df.itertuples(index=False), total=len(work_df), desc=f"Segmenting {candidate_params['candidate_id']}"):
        clip_id = sanitize_clip_id(row.clip_id)
        frame_file = Path(row.frame_file)

        payload = load_ssl_npz(
            npz_path=frame_file,
            fallback_duration_sec=safe_float(row.duration_sec, default=0.0),
            fallback_frame_hop_sec=safe_float(row.frame_hop_sec, default=0.02),
        )

        embeddings = payload["embeddings"]
        frame_times_sec = payload["frame_times_sec"]
        duration_sec = float(payload["duration_sec"])
        frame_hop_sec = float(payload["frame_hop_sec"])

        if embeddings.shape[0] == 0:
            segments_final = [(0, 0)]
            boundary_frames_initial = np.array([], dtype=np.int32)
            score_meta = {"median_score": 0.0, "mad_score": 0.0, "peak_height_threshold": 0.0}
            score_summary = {"score_mean": 0.0, "score_std": 0.0, "score_max": 0.0}
        else:
            embeddings_smoothed = smooth_embeddings(embeddings, int(candidate_params["smoothing_window_frames"]))
            change_scores = compute_change_scores(embeddings_smoothed, metric=str(candidate_params.get("distance_metric", "cosine")).lower())
            change_scores = smooth_scores(change_scores, int(candidate_params["change_score_smoothing_frames"]))

            boundary_frames_initial, score_meta = detect_boundary_frames(
                scores=change_scores,
                frame_hop_sec=frame_hop_sec,
                peak_prominence=float(candidate_params["peak_prominence"]),
                peak_height_mad_multiplier=float(candidate_params["peak_height_mad_multiplier"]),
                min_peak_distance_sec=float(candidate_params["min_peak_distance_sec"]),
                max_segments_per_clip=candidate_params.get("max_segments_per_clip", None),
            )

            segments_initial = boundaries_to_segments(num_frames=embeddings.shape[0], boundary_frames=boundary_frames_initial)
            segments_final = merge_short_segments(
                embeddings_smoothed=embeddings_smoothed,
                segments=segments_initial,
                frame_hop_sec=frame_hop_sec,
                min_segment_duration_sec=float(candidate_params["min_segment_duration_sec"]),
            )

            if not segments_final:
                segments_final = [(0, embeddings.shape[0])]

            score_summary = summarize_scores(change_scores)

        segment_rows.extend(
            build_segment_rows(
                clip_id=clip_id,
                frame_file=str(frame_file),
                duration_sec=duration_sec,
                frame_hop_sec=frame_hop_sec,
                frame_times_sec=frame_times_sec,
                segments=segments_final,
            )
        )

        summary_rows.append(
            build_summary_row(
                clip_id=clip_id,
                frame_file=str(frame_file),
                duration_sec=duration_sec,
                num_total_frames=int(embeddings.shape[0]),
                frame_hop_sec=frame_hop_sec,
                boundary_frames_initial=boundary_frames_initial,
                segments_final=segments_final,
                score_meta=score_meta,
                score_summary=score_summary,
            )
        )

    segments_df = pd.DataFrame(segment_rows)
    summary_df = pd.DataFrame(summary_rows)

    if not segments_df.empty:
        segments_df = segments_df.sort_values(["clip_id", "segment_index"], kind="stable").reset_index(drop=True)
    if not summary_df.empty:
        summary_df = summary_df.sort_values(["clip_id"], kind="stable").reset_index(drop=True)

    return segments_df, summary_df

In [ ]:
assert FRAME_INDEX_PATH.exists(), f"Missing frame index: {FRAME_INDEX_PATH}"
assert BASELINE_SEGMENTS_CSV.exists(), f"Missing baseline segments: {BASELINE_SEGMENTS_CSV}"
assert BASELINE_SUMMARY_CSV.exists(), f"Missing baseline summary: {BASELINE_SUMMARY_CSV}"

frame_index_df = load_frame_index(FRAME_INDEX_PATH)
processed_like = frame_index_df["status"].isin(["processed", "skipped_existing"])
work_df = frame_index_df.loc[processed_like].copy()

max_clips = SEG_CFG.get("max_clips", None)
if max_clips is not None:
    work_df = work_df.head(int(max_clips)).copy()

missing_npz_mask = ~work_df["frame_file"].map(lambda p: Path(p).exists())
if missing_npz_mask.any():
    missing_count = int(missing_npz_mask.sum())
    display(work_df.loc[missing_npz_mask, ["clip_id", "frame_file"]].head(10))
    raise FileNotFoundError(f"{missing_count} frame files listed in frame_index.csv are missing.")

baseline_segments_df = pd.read_csv(BASELINE_SEGMENTS_CSV)
baseline_summary_df = pd.read_csv(BASELINE_SUMMARY_CSV)

baseline_effective_params = {
    "candidate_id": "baseline_reference",
    "candidate_label": "current_baseline",
    "notes": "Current output/02_segments baseline preserved exactly as-is.",
    "distance_metric": str(SEG_CFG.get("distance_metric", "cosine")).lower(),
    "smoothing_window_frames": int(SEG_CFG.get("smoothing_window_frames", 5)),
    "change_score_smoothing_frames": int(SEG_CFG.get("change_score_smoothing_frames", 3)),
    "peak_prominence": float(SEG_CFG.get("peak_prominence", 0.025)),
    "peak_height_mad_multiplier": float(SEG_CFG.get("peak_height_mad_multiplier", 3.0)),
    "min_peak_distance_sec": float(SEG_CFG.get("min_peak_distance_sec", 0.20)),
    "min_segment_duration_sec": float(SEG_CFG.get("min_segment_duration_sec", 0.25)),
    "max_segments_per_clip": SEG_CFG.get("max_segments_per_clip", None),
}

baseline_metrics = summarize_candidate_outputs(
    segments_df=baseline_segments_df,
    summary_df=baseline_summary_df,
    candidate_params=baseline_effective_params,
)

with open(BASELINE_REFERENCE_JSON, "w", encoding="utf-8") as f:
    json.dump(baseline_metrics, f, indent=2)

print("Rows in frame_index.csv :", len(frame_index_df))
print("Rows ready for sweep    :", len(work_df))
print("Baseline clips          :", baseline_metrics["n_clips"])
print("Baseline segments       :", baseline_metrics["n_segments_total"])
print("Baseline single-seg %   :", round(baseline_metrics["single_segment_clip_pct"], 3))
display(pd.DataFrame([baseline_metrics]).T.rename(columns={0: "baseline_value"}))

logger.info("Loaded baseline metrics from %s", BASELINE_SEGMENTS_DIR)

In [ ]:
candidate_rows: List[Dict[str, Any]] = []
for item in SWEEP_CFG["default_candidates"]:
    row = deepcopy(item)
    row["distance_metric"] = str(row.get("distance_metric", SEG_CFG.get("distance_metric", "cosine"))).lower()
    row["source_notebook"] = "02b_segment_sweep.ipynb"
    row["created_at_utc"] = datetime.now(timezone.utc).isoformat()
    candidate_rows.append(row)

sweep_plan_df = pd.DataFrame(candidate_rows)

required_plan_cols = [
    "candidate_id",
    "candidate_label",
    "notes",
    "distance_metric",
    "smoothing_window_frames",
    "change_score_smoothing_frames",
    "peak_prominence",
    "peak_height_mad_multiplier",
    "min_peak_distance_sec",
    "min_segment_duration_sec",
    "max_segments_per_clip",
]
missing_plan_cols = [c for c in required_plan_cols if c not in sweep_plan_df.columns]
if missing_plan_cols:
    raise ValueError(f"sweep plan missing required columns: {missing_plan_cols}")

if sweep_plan_df["candidate_id"].duplicated().any():
    raise ValueError("Duplicate candidate_id values found in sweep plan.")

sweep_plan_df.to_csv(SWEEP_PLAN_CSV, index=False)
display(sweep_plan_df)

logger.info("Saved sweep plan to %s", SWEEP_PLAN_CSV)

In [ ]:
def normalize_optional_positive_int(value):
    if value is None:
        return None

    try:
        if pd.isna(value):
            return None
    except Exception:
        pass

    try:
        value_float = float(value)
    except Exception:
        return None

    if not np.isfinite(value_float):
        return None

    return max(int(value_float), 1)

In [ ]:
overwrite_candidates = bool(SWEEP_CFG.get("overwrite_candidates", False))

result_rows: List[Dict[str, Any]] = []

for candidate in sweep_plan_df.to_dict(orient="records"):
    candidate = dict(candidate)

    candidate["max_segments_per_clip"] = normalize_optional_positive_int(
        candidate.get("max_segments_per_clip", None)
    )

    candidate_id = str(candidate["candidate_id"])
    candidate_dir = SWEEP_DIR / candidate_id
    paths = ensure_candidate_dirs(candidate_dir)

    already_has_outputs = (
        paths["segments_csv"].exists() and paths["segment_summary_csv"].exists()
    )

    if already_has_outputs and not overwrite_candidates:
        logger.info("Candidate %s already exists. Loading saved outputs.", candidate_id)
        segments_df = pd.read_csv(paths["segments_csv"])
        summary_df = pd.read_csv(paths["segment_summary_csv"])
    else:
        logger.info("Running candidate %s", candidate_id)
        t0 = time.time()

        segments_df, summary_df = run_single_candidate(
            work_df=work_df,
            candidate_params=candidate,
        )

        segments_df.to_csv(paths["segments_csv"], index=False)
        summary_df.to_csv(paths["segment_summary_csv"], index=False)

        logger.info("Candidate %s completed in %.2f sec", candidate_id, time.time() - t0)

    metrics = summarize_candidate_outputs(
        segments_df=segments_df,
        summary_df=summary_df,
        candidate_params=candidate,
    )

    metrics["delta_single_segment_clip_pct_vs_baseline"] = float(
        metrics["single_segment_clip_pct"] - baseline_metrics["single_segment_clip_pct"]
    )
    metrics["delta_mean_segments_per_clip_vs_baseline"] = float(
        metrics["mean_segments_per_clip"] - baseline_metrics["mean_segments_per_clip"]
    )
    metrics["delta_tiny_segments_lt_015_pct_vs_baseline"] = float(
        metrics["tiny_segments_lt_015_pct"] - baseline_metrics["tiny_segments_lt_015_pct"]
    )
    metrics["baseline_reference_single_segment_clip_pct"] = float(
        baseline_metrics["single_segment_clip_pct"]
    )
    metrics["baseline_reference_mean_segments_per_clip"] = float(
        baseline_metrics["mean_segments_per_clip"]
    )

    with open(paths["candidate_config_json"], "w", encoding="utf-8") as f:
        json.dump(candidate, f, indent=2)

    with open(paths["candidate_metrics_json"], "w", encoding="utf-8") as f:
        json.dump(metrics, f, indent=2)

    write_markdown(
        paths["notes_md"],
        [
            f"# {candidate_id}",
            "",
            f"- Label: {candidate['candidate_label']}",
            f"- Notes: {candidate['notes']}",
            f"- Segments CSV: {paths['segments_csv'].name}",
            f"- Segment summary CSV: {paths['segment_summary_csv'].name}",
            f"- Downstream folders prepared for 02c: 03_segment_vectors/, 04_kstudy/, 05_tokens/, 06_dictionary/, 07_audits/",
            f"- Baseline single-segment clip %: {baseline_metrics['single_segment_clip_pct']:.4f}",
            f"- Candidate single-segment clip %: {metrics['single_segment_clip_pct']:.4f}",
            f"- Candidate max_segments_per_clip: {candidate['max_segments_per_clip']}",
        ],
    )

    result_rows.append(metrics)

sweep_results_df = (
    pd.DataFrame(result_rows)
    .sort_values(
        ["single_segment_clip_pct", "tiny_segments_lt_015_pct", "mean_segments_per_clip"],
        ascending=[True, True, False],
    )
    .reset_index(drop=True)
)

sweep_results_df.to_csv(SWEEP_RESULTS_CSV, index=False)

display(
    sweep_results_df[
        [
            "candidate_id",
            "candidate_label",
            "single_segment_clip_pct",
            "delta_single_segment_clip_pct_vs_baseline",
            "mean_segments_per_clip",
            "delta_mean_segments_per_clip_vs_baseline",
            "tiny_segments_lt_015_pct",
            "delta_tiny_segments_lt_015_pct_vs_baseline",
            "p50_segment_duration_sec",
            "max_segments_in_any_clip",
        ]
    ]
)

logger.info("Saved sweep results to %s", SWEEP_RESULTS_CSV)

In [ ]:
if not sweep_results_df.empty:
    summary_view = sweep_results_df[
        [
            "candidate_id",
            "candidate_label",
            "n_clips",
            "n_segments_total",
            "mean_segments_per_clip",
            "median_segments_per_clip",
            "single_segment_clips",
            "single_segment_clip_pct",
            "tiny_segments_lt_015_pct",
            "p10_segment_duration_sec",
            "p50_segment_duration_sec",
            "p90_segment_duration_sec",
            "max_segments_in_any_clip",
        ]
    ].copy()
    display(summary_view)
else:
    print("No sweep results were generated.")

print("Expected next notebook: 02c_compare_segmentation_candidates.ipynb")
print("Sweep plan file       :", SWEEP_PLAN_CSV)
print("Sweep results file    :", SWEEP_RESULTS_CSV)
print("Baseline reference    :", BASELINE_REFERENCE_JSON)
print("Main log file         :", LOG_FILE)